# LangChain: Agents (Groq Llama 3.1)

## Outline
* **Built-in Tools** - DuckDuckGo search, Wikipedia
* **Python Agent** - Execute Python code dynamically
* **Custom Tools** - Define your own tools with @tool decorator
* **Debug Mode** - Inspect agent reasoning process

**Model Used:** Groq Llama-3.1-8b-instant

**Key Concept:** LLMs as **reasoning engines**, not just knowledge stores. Agents decide which tools to use and when to use them.

**What is an Agent?**
An agent uses an LLM to decide which actions to take and in what order. Unlike chains (predetermined sequence), agents dynamically choose their path based on inputs.


In [ ]:
# Cell 2: Install Required Packages

!pip install langchain langchain-groq python-dotenv wikipedia duckduckgo-search -q


In [ ]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings("ignore")

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")


In [ ]:
# Cell 4: Import Agent Components

from langchain.agents import load_tools, initialize_agent, AgentType
from langchain_groq import ChatGroq
from langchain.agents.agent_toolkits import create_python_agent
from langchain.tools.python.tool import PythonREPLTool

print("✅ Agent components imported")


In [ ]:
# Cell 5: Initialize LLM

# Set model
llm_model = "llama-3.1-8b-instant"

# Initialize Groq LLM
llm = ChatGroq(temperature=0, model=llm_model, groq_api_key=groq_api_key)

print(f"✅ Model initialized: {llm_model}")


In [ ]:
# Cell 6: Load Built-in Tools

# Load Wikipedia and math tools
tools = load_tools(["llm-math", "wikipedia"], llm=llm)

print(f"✅ Loaded {len(tools)} tools:")
for tool in tools:
    print(f"  - {tool.name}")


In [ ]:
# Cell 7: Create Agent

# Initialize agent with tools
agent = initialize_agent(
    tools, 
    llm, 
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    handle_parsing_errors=True,
    verbose=True
)

print("✅ Agent created successfully")
print("\nAgent Type: CHAT_ZERO_SHOT_REACT_DESCRIPTION")
print("  - CHAT: Optimized for chat models")
print("  - ZERO_SHOT: No examples needed")
print("  - REACT: Reasoning + Acting prompting strategy")


In [ ]:
# Cell 8: Test Math Tool

# Simple math question
response = agent.run("What is the 25% of 300?")

print("\nFinal Answer:")
print(response)


In [ ]:
# Cell 9: Test Wikipedia Tool

question = "Tom M. Mitchell is an American computer scientist \
and the Founders University Professor at Carnegie Mellon University (CMU) \
what book did he write?"

result = agent.run(question)

print("\nFinal Answer:")
print(result)


In [ ]:
# Cell 10: Test Current Events Question

# Ask about 2022 World Cup (after training cutoff)
# Note: Agent will attempt to search but results may vary
question = "Who won the 2022 FIFA World Cup?"

try:
    result = agent.run(question)
    print("\nFinal Answer:")
    print(result)
except Exception as e:
    print(f"Agent encountered an issue: {e}")
    print("\n⚠️ Agents are experimental and may not always succeed")


## Python Agent

The **Python Agent** can write and execute Python code to solve problems. This is powerful for:
- Data manipulation
- Mathematical calculations
- List/array operations
- Any task that's easier to solve with code than natural language


In [ ]:
# Cell 11: Create Python Agent

agent = create_python_agent(
    llm,
    tool=PythonREPLTool(),
    verbose=True
)

print("✅ Python Agent created")
